In [6]:
#!git clone https://github.com/yuanzhumia0/CA1-2026025-Machine-Learning-for-Business.git

Cloning into 'CA1-2026025-Machine-Learning-for-Business'...


### Time-Based Data Split

The dataset was split based on time to avoid data leakage.

- **January to June:** Training set
- **July to September:** Validation set
- **October to December:** Test set

The training set is used to train the model. The validation set is used to compare models and tune parameters. The test set is used at the end to evaluate the final model on unseen future data.

In [2]:
#%pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

df = pd.read_parquet(
    "/Users/cat/Documents/GitHub/CA1-2026025-Machine-Learning-for-Business/yellow_tripdata_2025-01.parquet"
)


Check how many rows 

In [5]:
df.head() 

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


Check the number of columns.

In [7]:
df.shape[1]

20

Get all column names of the DataFrame.

In [8]:
df.columns.tolist()

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

### January Data Overview

The January dataset has 3,475,226 rows and 20 columns. Each row contains information about one taxi trip, such as the trip distance, pickup time, drop-off time, and fare.

The pickup and drop-off times are already in datetime format. I can use them later to calculate how long each trip lasted.

Next, I will check for missing values.

### Check missing values in each column

In [10]:
missing_values = pd.DataFrame({
    "Missing values": df.isna().sum(),
    "Missing (%)": (df.isna().mean() * 100).round(2)
})

missing_values

,Missing values,Missing (%)
VendorID,0,0.00
tpep_pickup_datetime,0,0.00
tpep_dropoff_datetime,0,0.00
passenger_count,540149,15.54
trip_distance,0,0.00
RatecodeID,540149,15.54
store_and_fwd_flag,540149,15.54
PULocationID,0,0.00
DOLocationID,0,0.00
payment_type,0,0.00


### Missing Values

The result shows that most columns do not have missing values.

However, `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, and `Airport_fee` have missing values.

Each of these columns has **540,149 missing values**, which is **15.54%** of the total data.

These missing values may need to be handled before further analysis.

In [11]:
# Summary statistics for numerical columns
df.describe().T

,count,mean,min,25%,50%,75%,max,std
VendorID,3475226.0,1.785428,1.0,2.0,2.0,2.0,7.0,0.426328
tpep_pickup_datetime,3475226,2025-01-17 11:02:55.910964,2024-12-31 20:47:55,2025-01-10 07:59:01,2025-01-17 15:41:33,2025-01-24 19:34:06,2025-02-01 00:00:44,NaN
tpep_dropoff_datetime,3475226,2025-01-17 11:17:56.997901,2024-12-18 07:52:40,2025-01-10 08:15:29.500000,2025-01-17 15:59:34,2025-01-24 19:48:31,2025-02-01 23:44:11,NaN
passenger_count,2935077.0,1.297859,0.0,1.0,1.0,1.0,9.0,0.75075
trip_distance,3475226.0,5.855126,0.0,0.98,1.67,3.1,276423.57,564.6016
RatecodeID,2935077.0,2.482535,1.0,1.0,1.0,1.0,99.0,11.632772
PULocationID,3475226.0,165.191576,1.0,132.0,162.0,234.0,265.0,64.529483
DOLocationID,3475226.0,164.125177,1.0,113.0,162.0,234.0,265.0,69.401686
payment_type,3475226.0,1.036623,0.0,1.0,1.0,1.0,5.0,0.701333
fare_amount,3475226.0,17.081803,-900.0,8.6,12.11,19.5,863372.12,463.472918


### Summary Statistics

The summary statistics show the main features of the numerical data.

The average passenger count is about **1.30**, and the average trip distance is about **5.86 miles**.

The average fare amount is about **17.08**, while the average total amount is about **25.61**.

Some very large, small, and negative values can also be seen in the data. These may be outliers and should be checked before further analysis.

In [12]:
# Check for duplicate rows
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


### Duplicate Records¶

I checked for rows with identical values in all columns. No duplicate rows were found, so no rows were removed in this step.

In [13]:
# Keep trips that started in January 2025
df_clean = df[
    (df["tpep_pickup_datetime"] >= "2025-01-01") &
    (df["tpep_pickup_datetime"] < "2025-02-01")
].copy()

print("Rows before filtering:", len(df))
print("Rows after filtering:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

Rows before filtering: 3475226
Rows after filtering: 3475204
Rows removed: 22


### Selecting January Trips

I kept trips with pickup times in January 2025. This removed 22 records and left 3,475,204 trips for further analysis.

In [14]:
# Calculate trip duration in minutes
df_clean["trip_duration_minutes"] = (
    df_clean["tpep_dropoff_datetime"] -
    df_clean["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# Display summary statistics
print(df_clean["trip_duration_minutes"].describe())

# Count trips with zero or negative duration
print(
    "\nTrips with duration <= 0:",
    (df_clean["trip_duration_minutes"] <= 0).sum()
)

count    3.475204e+06
mean     1.501811e+01
std      3.871367e+01
min     -5.147232e+04
25%      7.283333e+00
50%      1.170000e+01
75%      1.833333e+01
max      5.626317e+03
Name: trip_duration_minutes, dtype: float64

Trips with duration <= 0: 2051


### Trip Duration

I calculated the trip duration in minutes using the pickup and drop-off times.

The average trip duration is about **15.02 minutes**, and the median duration is about **11.7 minutes**.

There are **2,051 trips** with zero or negative duration. These records are not valid, so they will be removed during data cleaning.

There are also some trips with very long durations, which may be outliers and need further checking.

In [15]:
# Remove trips with zero or negative duration
rows_before = len(df_clean)

df_clean = df_clean[
    df_clean["trip_duration_minutes"] > 0
].copy()

print("Rows removed:", rows_before - len(df_clean))
print("Rows remaining:", len(df_clean))

Rows removed: 2051
Rows remaining: 3473153


In [16]:
# Check the upper percentiles of trip duration
df_clean["trip_duration_minutes"].quantile([0.95, 0.99, 0.999])

0.950    36.216667
0.990    58.916667
0.999    98.280800
Name: trip_duration_minutes, dtype: float64

### Checking Long Trips

I removed 2,051 trips with zero or negative duration, leaving 3,473,153 records.

About 99% of trips lasted no more than 58.92 minutes, and 99.9% lasted no more than 98.28 minutes. Longer trips may be unusual, but they are not necessarily errors. I will check the longest trips before deciding whether to remove them.

In [17]:
# Inspect the 10 longest trips
columns_to_check = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_minutes",
    "trip_distance",
    "fare_amount",
    "total_amount"
]

df_clean.nlargest(10, "trip_duration_minutes")[columns_to_check]

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_minutes,trip_distance,fare_amount,total_amount
1858816,2025-01-21 12:23:41,2025-01-25 10:10:00,5626.316667,6.55,-28.20,-36.45
1858817,2025-01-21 12:23:41,2025-01-25 10:10:00,5626.316667,6.55,28.20,36.45
414374,2025-01-06 02:12:11,2025-01-09 13:54:19,5022.133333,91.51,336.90,339.40
2460536,2025-01-27 11:58:55,2025-01-30 09:42:55,4184.000000,0.00,3.00,4.50
898201,2025-01-11 08:04:00,2025-01-13 16:49:28,3405.466667,0.04,70.00,76.50
2147536,2025-01-24 07:07:21,2025-01-26 10:09:44,3062.383333,0.00,3.00,4.50
85681,2025-01-02 09:40:31,2025-01-04 09:37:01,2876.500000,0.00,3.00,4.50
1536147,2025-01-17 16:45:49,2025-01-19 16:40:59,2875.166667,1.98,12.80,20.05
556772,2025-01-07 18:07:33,2025-01-09 16:12:05,2764.533333,1.86,47.80,51.80
3417048,2025-01-29 21:39:23,2025-01-31 16:48:43,2589.333333,7.52,31.02,35.77


### Removing Extremely Long Trips

Some of the longest records lasted several days but had short or zero trip distances. These records appear unsuitable for analysing typical taxi trips.

For this project, I will keep trips lasting up to 180 minutes. This is a practical limit for my analysis, although it may exclude some valid long trips.

In [18]:
# Keep trips lasting up to 3 hours
rows_before = len(df_clean)

df_clean = df_clean[
    df_clean["trip_duration_minutes"] <= 180
].copy()

print("Rows removed:", rows_before - len(df_clean))
print("Rows remaining:", len(df_clean))

Rows removed: 1377
Rows remaining: 3471776


After applying the 180-minute limit, I removed 1,377 records. The dataset now contains 3,471,776 trips, each with a duration greater than zero and no more than 180 minutes.

In [19]:
# Keep trips starting in January
df_clean = df[
    (df["tpep_pickup_datetime"] >= "2025-01-01") &
    (df["tpep_pickup_datetime"] < "2025-02-01")
].copy()

# Calculate trip duration in minutes
df_clean["trip_duration_minutes"] = (
    df_clean["tpep_dropoff_datetime"] -
    df_clean["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# Keep durations greater than 0 and no more than 180 minutes
df_clean = df_clean[
    (df_clean["trip_duration_minutes"] > 0) &
    (df_clean["trip_duration_minutes"] <= 180)
].copy()

print("Rows remaining:", len(df_clean))

Rows remaining: 3471776


### Cleaning Trip Duration

I kept trips that started in January 2025 and calculated their duration in minutes.

I removed 2,051 trips with zero or negative duration and 1,377 trips lasting more than 180 minutes. After these steps, 3,471,776 records remain.

Next, I will check trip distances and fares for unusual values.

In [20]:
# Check non-positive distances and fares
print("Trips with distance <= 0:",
      (df_clean["trip_distance"] <= 0).sum())

print("Trips with fare <= 0:",
      (df_clean["fare_amount"] <= 0).sum())

print("Trips with total amount <= 0:",
      (df_clean["total_amount"] <= 0).sum())

# Check percentiles and maximum values
df_clean[["trip_distance", "fare_amount", "total_amount"]].quantile(
    [0.50, 0.95, 0.99, 0.999, 1.0]
)

Trips with distance <= 0: 90172
Trips with fare <= 0: 145441
Trips with total amount <= 0: 63535


,trip_distance,fare_amount,total_amount
0.500,1.67,12.11,19.95000
0.950,11.83,52.00,74.00000
0.990,19.50,72.30,102.92000
0.999,28.82,131.10,163.24225
1.000,276423.57,863372.12,863380.37000


### Checking Trip Distance and Fares

There are 90,172 records with non-positive trip distances, 145,441 with non-positive fares, and 63,535 with non-positive total amounts. Some records may appear in more than one group.

For this project, I will focus on trips with positive distances, fares, and total amounts. This excludes zero-distance records and non-positive charges, which may include refunds or adjustments.

The maximum distance and fare are also much higher than the 99.9th percentile values. I will examine these extreme values separately.

In [21]:
# Keep trips with positive distances and charges
rows_before = len(df_clean)

df_clean = df_clean[
    (df_clean["trip_distance"] > 0) &
    (df_clean["fare_amount"] > 0) &
    (df_clean["total_amount"] > 0)
].copy()

print("Rows removed:", rows_before - len(df_clean))
print("Rows remaining:", len(df_clean))

Rows removed: 220634
Rows remaining: 3251142


### Removing Non-Positive Distances and Charges

I kept trips with positive trip distances, fares, and total amounts. This removed 220,634 records, leaving 3,251,142 trips.

The remaining data still needs to be checked for extremely large distances and fares.

In [22]:
# Inspect the 10 longest distances
df_clean.nlargest(10, "trip_distance")[columns_to_check]

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_minutes,trip_distance,fare_amount,total_amount
3188501,2025-01-19 14:51:00,2025-01-19 14:58:00,7.0,276099.95,9.13,13.88
3240337,2025-01-21 09:19:00,2025-01-21 09:45:00,26.0,222167.49,31.19,35.94
3112173,2025-01-16 10:47:00,2025-01-16 11:11:00,24.0,206137.99,24.89,29.64
3374854,2025-01-26 17:46:00,2025-01-26 17:59:00,13.0,202771.63,10.10,14.85
3081597,2025-01-14 19:18:00,2025-01-14 19:36:00,18.0,189687.43,12.70,17.45
3069039,2025-01-13 21:11:00,2025-01-13 21:17:00,6.0,181139.99,6.33,11.08
3138510,2025-01-17 14:25:00,2025-01-17 14:40:00,15.0,164959.95,14.05,18.05
2956166,2025-01-03 15:47:00,2025-01-03 16:06:00,19.0,158925.09,10.82,14.82
3009250,2025-01-10 06:58:00,2025-01-10 07:18:00,20.0,156037.94,28.74,33.49
2993436,2025-01-09 06:18:00,2025-01-09 06:24:00,6.0,143712.27,8.51,12.51


In [23]:
# Inspect the 10 highest fares
df_clean.nlargest(10, "fare_amount")[columns_to_check]

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_minutes,trip_distance,fare_amount,total_amount
1780915,2025-01-20 12:07:18,2025-01-20 12:12:42,5.400000,1.60,863372.12,863380.37
1346041,2025-01-15 19:14:42,2025-01-15 21:46:13,151.516667,143.54,936.80,969.05
562143,2025-01-07 19:12:25,2025-01-07 19:14:04,1.650000,0.10,900.00,901.00
1126773,2025-01-13 17:19:58,2025-01-13 20:07:08,167.166667,133.30,893.75,896.50
74028,2025-01-01 23:13:43,2025-01-02 01:47:38,153.916667,132.27,826.20,865.39
34810,2025-01-01 12:53:37,2025-01-01 15:25:57,152.333333,119.66,773.00,794.82
1160453,2025-01-14 02:08:52,2025-01-14 02:10:20,1.466667,0.35,700.00,704.25
160259,2025-01-03 06:30:10,2025-01-03 08:17:18,107.133333,111.20,670.10,702.11
899076,2025-01-11 09:50:17,2025-01-11 12:43:04,172.783333,78.94,634.40,653.21
680160,2025-01-08 23:54:59,2025-01-09 01:48:58,113.983333,95.59,607.10,640.98


### Checking Extreme Values

Some records have distances that are extremely large despite short trip durations. One trip also has a fare of 863,372.12 for a distance of only 1.60.

These values appear inconsistent with the other trip information. However, some high fares are associated with longer trips, so I will not remove all high-fare records automatically.

In [24]:
# Remove extremely large distances and charges
rows_before = len(df_clean)

df_clean = df_clean[
    (df_clean["trip_distance"] <= 1000) &
    (df_clean["fare_amount"] <= 1000) &
    (df_clean["total_amount"] <= 1000)
].copy()

print("Rows removed:", rows_before - len(df_clean))
print("Rows remaining:", len(df_clean))

Rows removed: 96
Rows remaining: 3251046


### Removing Extreme Values

I removed records where the trip distance, fare, or total amount exceeded 1,000. This removed 96 records, leaving 3,251,046 trips.

This was an initial check for extreme values. Further checks are still needed.

In [25]:
# Calculate distance travelled per hour
df_clean["distance_per_hour"] = (
    df_clean["trip_distance"] /
    (df_clean["trip_duration_minutes"] / 60)
)

# Inspect the highest values
df_clean.nlargest(10, "distance_per_hour")[
    columns_to_check + ["distance_per_hour"]
]

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_minutes,trip_distance,fare_amount,total_amount,distance_per_hour
3012886,2025-01-10 13:01:23,2025-01-10 13:01:24,0.016667,18.78,2.90,60.20,67608.0
3405685,2025-01-29 07:01:14,2025-01-29 07:01:15,0.016667,16.27,1.45,53.05,58572.0
3382390,2025-01-27 12:01:56,2025-01-27 12:01:57,0.016667,13.22,2.90,39.15,47592.0
1567952,2025-01-17 21:59:39,2025-01-17 21:59:40,0.016667,8.12,32.40,51.02,29232.0
3083790,2025-01-14 21:57:05,2025-01-14 21:57:06,0.016667,7.12,32.92,37.67,25632.0
3410352,2025-01-29 16:01:18,2025-01-29 16:01:20,0.033333,10.90,2.90,31.68,19620.0
771923,2025-01-09 21:17:26,2025-01-09 21:17:29,0.050000,13.80,3.00,8.75,16560.0
727110,2025-01-09 15:47:15,2025-01-09 15:47:19,0.066667,17.60,70.00,74.75,15840.0
2950978,2025-01-02 00:01:43,2025-01-02 00:01:45,0.033333,8.65,2.90,33.30,15570.0
402091,2025-01-05 20:29:32,2025-01-05 20:29:35,0.050000,12.50,70.00,74.75,15000.0


### Checking Distance and Duration

Some trips have durations of only a few seconds but large distances. This produces extremely high average speeds and suggests problems with the recorded distance or timestamps.

I will apply an upper speed limit as a practical cleaning rule for this project.

In [26]:
# Check the distribution of average speed
print(df_clean["distance_per_hour"].quantile(
    [0.50, 0.95, 0.99, 0.999]
))

# Count records above the proposed threshold
print(
    "\nTrips with distance per hour > 100:",
    (df_clean["distance_per_hour"] > 100).sum()
)

0.500     9.625668
0.950    25.144729
0.990    35.481834
0.999    46.842742
Name: distance_per_hour, dtype: float64

Trips with distance per hour > 100: 743


### Checking Average Speed

I calculated average speed using trip distance in miles and duration in hours. About 99.9% of trips have an average speed below 46.84 mph.

There are 743 records with an average speed above 100 mph. I will remove these records using 100 mph as a practical cleaning limit for this project.

In [27]:
# Remove trips with average speeds above 100 mph
rows_before = len(df_clean)

df_clean = df_clean[
    df_clean["distance_per_hour"] <= 100
].copy()

print("Rows removed:", rows_before - len(df_clean))
print("Rows remaining:", len(df_clean))

Rows removed: 743
Rows remaining: 3250303


 ### I removed 743 records with average speeds above 100 mph. After this step, 3,250,303 trips remain in the dataset.

In [28]:
# Check missing values after cleaning
missing_after_cleaning = pd.DataFrame({
    "Missing values": df_clean.isna().sum(),
    "Missing (%)": (df_clean.isna().mean() * 100).round(2)
})

missing_after_cleaning

,Missing values,Missing (%)
VendorID,0,0.0
tpep_pickup_datetime,0,0.0
tpep_dropoff_datetime,0,0.0
passenger_count,412942,12.7
trip_distance,0,0.0
RatecodeID,412942,12.7
store_and_fwd_flag,412942,12.7
PULocationID,0,0.0
DOLocationID,0,0.0
payment_type,0,0.0


### Missing Values After Cleaning

After cleaning, five columns still have 412,942 missing values each, representing 12.70% of the remaining records.

Trip distance, fare, total amount, trip duration, and average speed have no missing values. I will check passenger counts before deciding how to handle their missing values.

In [29]:
# Check passenger counts, including missing values
df_clean["passenger_count"].value_counts(dropna=False).sort_index()

passenger_count
0.0      23576
1.0    2248389
2.0     393536
3.0      87750
4.0      54738
5.0      17537
6.0      11825
7.0          1
8.0          6
9.0          3
NaN     412942
Name: count, dtype: int64

### Selecting Analysis Columns

Most trips with a recorded passenger count have one passenger. However, passenger_count has 412,942 missing values and 23,576 zero values.

I will exclude passenger_count from the analysis table rather than guess the missing counts. I will keep the trip records and select columns with no missing values for further analysis.

In [30]:
# Select columns for further analysis
analysis_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "trip_duration_minutes",
    "distance_per_hour"
]

df_january = df_clean[analysis_columns].copy()

In [31]:
print("Final shape:", df_january.shape)
print("\nMissing values:")
print(df_january.isna().sum())

df_january[
    ["trip_distance", "fare_amount", "total_amount",
     "trip_duration_minutes", "distance_per_hour"]
].describe().T

Final shape: (3250303, 10)

Missing values:
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
PULocationID             0
DOLocationID             0
payment_type             0
trip_distance            0
fare_amount              0
total_amount             0
trip_duration_minutes    0
distance_per_hour        0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
trip_distance,3250303.0,3.167419,4.156297,0.010000,1.010000,1.700000,3.140000,150.110000
fare_amount,3250303.0,17.949531,16.315954,0.010000,8.600000,12.800000,19.800000,936.800000
total_amount,3250303.0,26.812143,20.489129,1.010000,15.950000,20.350000,28.250000,969.050000
trip_duration_minutes,3250303.0,14.679015,11.349437,0.033333,7.333333,11.700000,18.333333,179.966667
distance_per_hour,3250303.0,11.335741,6.437421,0.007980,7.391635,9.624765,13.026648,99.403950


### January Cleaning Summary

The original January file contained 3,475,226 records. After applying the date, duration, distance, charge, and speed filters, 3,250,303 records remain.

I selected 10 columns with no missing values and saved the January analysis table. These filters define the trips included in this project, but they do not guarantee that every remaining record is error-free.